# Stage 9b — KenLM-equivalent char 4-gram LM + CTC prefix beam search (Kaggle T4)

Drops a 4-gram character LM (Kneser-Ney smoothing, pure Python — no KenLM compilation required) on top of the **existing Stage 9a checkpoints**.  No retraining.

Decoding combines three paths per clip:
1. CTC greedy decode (existing).
2. Attention greedy decode (existing).
3. **CTC prefix beam search with LM shallow fusion** (new):
   `score = log p_ctc + alpha · log p_lm + beta · |prefix|`

Headline = best of the three per clip.  Per-fold LM is trained ONLY on that fold's TRAIN signers (no val leakage).

**Pass/fail vs Stage 9a (0.4889 full / 0.4775 stripped)**:
- Headline pass: any improvement vs Stage 9a on the same fold.
- Stretch:        full ≤ 0.45 (a further 4 pt reduction on top of 9a).

**Wall-clock on Kaggle T4 (one session)**:

| Phase | Time |
|---|---|
| LM training (5 folds × pure Python) | ~3 min |
| Beam search eval (5 folds × ~720 val clips × beam=32) | ~30 min |
| Total | **~35 min** |

Tiny budget vs the Stage 9a sweep because there's no training.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'CUDA available : {torch.cuda.is_available()}')

## Cell 2 — Locate artifacts

Attach your committed Stage 9a kernel via **Data → Add Data → Notebook output**.  Cell 2 finds the checkpoints, cache, and manifest under `/kaggle/input/**` regardless of nesting.

In [ ]:
import os, glob

def _findall(pattern: str):
    return sorted(
        glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
        + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True)
    )
def _first(pattern):
    m = _findall(pattern)
    return m[0] if m else None

CACHE_PATH   = _first('skeleton_features_t32.pt')
CV_MANIFEST  = _first('subject_cv5.json')
STAGE9A_CKPTS = _findall('stage9a_fold*_stage9a_best.pt')

print(f'cache       : {CACHE_PATH}')
print(f'manifest    : {CV_MANIFEST}')
print(f'stage9a ckpts ({len(STAGE9A_CKPTS)}):')
for p in STAGE9A_CKPTS:
    print(f'  {p}')
assert CACHE_PATH and CV_MANIFEST, 'Cache or manifest missing — attach the Stage 9a kernel output.'
assert len(STAGE9A_CKPTS) == 5, f'Expected 5 Stage 9a checkpoints, got {len(STAGE9A_CKPTS)}'

LM_DIR = '/kaggle/working/lms'
RESULTS_PATH = '/kaggle/working/stage9b_results.json'
os.makedirs(LM_DIR, exist_ok=True)

## Cell 3 — Train per-fold char 4-gram LM (WiTA labels, train signers only)

Trains on each fold's TRAIN-set labels — never sees val signers.  ~30 sec per fold (3 minutes total).

In [ ]:
import re
ORDER = 4
CORPUS = 'wita_labels'   # set to 'external' + --external-text for an LM trained on Wikipedia etc.

lm_paths = {}
for ckpt in STAGE9A_CKPTS:
    m = re.search(r'fold(\d+)', os.path.basename(ckpt))
    fold = int(m.group(1))
    out_lm = os.path.join(LM_DIR, f'char_lm_{ORDER}gram_{CORPUS}_fold{fold}.pkl')
    if os.path.exists(out_lm):
        print(f'  fold {fold}: LM exists at {out_lm}')
    else:
        print(f'  fold {fold}: training LM...')
        !python /kaggle/working/wita_v2/scripts/train_char_lm.py \
            --skeleton-cache {CACHE_PATH} \
            --cv-manifest    {CV_MANIFEST} \
            --fold           {fold} \
            --order          {ORDER} \
            --corpus         {CORPUS} \
            --out            {out_lm}
    lm_paths[fold] = out_lm
print(f'\nAll LMs ready: {lm_paths}')

## Cell 4 — Run Stage 9b: CTC + LM beam search + attention rescoring

alpha = LM weight (typical 0.3–1.5).  Try 0.5 first.
beta  = length bonus (counters LM's bias toward short outputs).  0.0 default.
beam  = beam width.  32 is the sweet spot for char LMs; 64 if you have time.

In [ ]:
ALPHA = 0.5
BETA  = 0.0
BEAM  = 32

# Pick the parent dirs of the ckpts / LMs so the globs match what the eval script expects.
CKPT_GLOB = '/kaggle/input/**/stage9a_fold*_stage9a_best.pt'
LM_GLOB   = os.path.join(LM_DIR, f'char_lm_{ORDER}gram_{CORPUS}_fold*.pkl')

!python /kaggle/working/wita_v2/scripts/eval_stage9b.py \
    --skeleton-cache {CACHE_PATH} \
    --cv-manifest    {CV_MANIFEST} \
    --checkpoint-glob "{CKPT_GLOB}" \
    --lm-glob        "{LM_GLOB}" \
    --out            {RESULTS_PATH} \
    --alpha          {ALPHA} \
    --beta           {BETA} \
    --beam           {BEAM}

## Cell 5 — Aggregate (dual-cohort) + verdict

In [ ]:
import json, numpy as np
from wita_v2.reports.template.stripped_cohort import dual_cohort_summary

with open(RESULTS_PATH) as f:
    all_r = json.load(f)

print(' fold    CER ctc     CER attn    CER beam    CER best')
for r in sorted(all_r, key=lambda x: x['fold']):
    print(f'  {r["fold"]:>2d}    {r["cer_ctc"]:.4f}      '
          f'{r["cer_attn"]:.4f}      {r["cer_beam"]:.4f}      {r["best_val_cer"]:.4f}')

s = dual_cohort_summary(RESULTS_PATH, 'stage9b')
print(f'\n     full cohort   : {s["full_mean"]:.4f} ± {s["full_std"]:.4f}')
print(f'     PHW/KIM-strip : {s["stripped_mean"]:.4f} ± {s["stripped_std"]:.4f}')
print(f'     Δ (full-strip): {s["delta_mean"]:+.4f}')

STAGE9A_FULL     = 0.4889
STAGE9A_STRIPPED = 0.4775
print('\n=== Stage 9b verdict ===')
print(f'  Stage 9a baseline (full / stripped) : {STAGE9A_FULL:.4f} / {STAGE9A_STRIPPED:.4f}')
print(f'  Stage 9b mean      (full / stripped) : {s["full_mean"]:.4f} / {s["stripped_mean"]:.4f}')
if s['full_mean'] <= 0.45:
    print('  ✅ STRETCH (≤ 0.45) — LM rescoring substantially helps.')
elif s['full_mean'] <= STAGE9A_FULL - 0.01:
    print(f'  ✅ HEADLINE — beats Stage 9a by {STAGE9A_FULL - s["full_mean"]:.4f}.')
elif abs(s['full_mean'] - STAGE9A_FULL) <= 0.01:
    print(f'  ⚖  TIE with Stage 9a. LM not adding value at alpha={ALPHA}, beta={BETA}.')
    print('    Sweep alpha/beta in Cell 4 and re-run; alpha=1.0 is a common upper bound.')
else:
    print(f'  ❌ REGRESS by {s["full_mean"] - STAGE9A_FULL:+.4f}. LM dominates; lower alpha or check corpus.')

## Cell 6 — Per-signer scatter (Stage 9b vs Stage 9a)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer_9b = {}
for r in all_r:
    per_signer_9b.update(r.get('best_per_signer_val_cer', {}) or {})
items = sorted(per_signer_9b.items(), key=lambda kv: kv[1])
xs = list(range(len(items)))
ys = [v for _, v in items]
DATASET_LIMIT = ['PHW', 'KIM']
MODEL_HARD    = ['PJH','SYB','KJM','KNY','LKS','YMG']
def _c(s):
    if s in DATASET_LIMIT: return '#7f7f7f'
    if s in MODEL_HARD:    return '#d62728'
    return '#1f77b4'
colors = [_c(s) for s,_ in items]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.scatter(xs, ys, s=30, c=colors)
ax.axhline(0.55, color='green', linestyle='--', alpha=0.5, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.5, label='hard ≥ 0.75')
ax.set_xticks(xs); ax.set_xticklabels([s for s,_ in items], rotation=90, fontsize=7)
ax.set_ylabel('Stage 9b val CER (best of CTC / attn / CTC+LM beam)')
ax.set_xlabel('signer (sorted by CER)')
ax.set_title(f'Stage 9b — per-signer val CER  '
             f'(alpha={ALPHA}, beam={BEAM})')
ax.legend(frameon=False); ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
os.makedirs('/kaggle/working/logs', exist_ok=True)
plt.savefig('/kaggle/working/logs/stage9b_per_signer_scatter.png', dpi=140)
plt.show()

## Cell 7 — Commit kernel

Save & Run All so the LMs, results, and scatter PNG survive:
- `/kaggle/working/stage9b_results.json`
- `/kaggle/working/lms/char_lm_4gram_*_fold*.pkl`
- `/kaggle/working/logs/stage9b_per_signer_scatter.png`

If the verdict was a TIE, try:
- alpha sweep: 0.3, 0.5, 0.8, 1.2 (re-run Cell 4 with new ALPHA, results JSON gets overwritten).
- beta sweep: 0.0, 0.5, 1.0 (counters LM's short-output bias).
- Larger beam: 64 or 128 (slower, sometimes finds better paths).
- Switch CORPUS in Cell 3 to 'external' with a generic English ARPA-style text file (one sentence per line, lowercased).